In [4]:
# ==========================================================
# 0. ИМПОРТЫ И БАЗОВЫЕ НАСТРОЙКИ
# ==========================================================
import os
import sys
import glob
import time
import torch
import chromadb
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

project_root = os.path.abspath('..')
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

from hypothesis_factory.ingestion_gpu import GpuDoclingReader

load_dotenv()
API_KEY = os.getenv("YANDEX_API_KEY")
FOLDER_ID = os.getenv("YANDEX_FOLDER_ID")
if not API_KEY or not FOLDER_ID:
    print(" ВНИМАНИЕ: Ключи Yandex API не найдены! Убедитесь, что .env настроен.")

# ==========================================================
# 1. ПРОВЕРКА GPU И МОДЕЛЬ ЭМБЕДДИНГОВ
# ==========================================================
print("=" * 60)
print("ПРОВЕРКА GPU И ЗАГРУЗКА МОДЕЛЕЙ")
print("=" * 60)
cuda_ok = torch.cuda.is_available()
if cuda_ok:
    print(f"Устройство: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA НЕДОСТУПНА. Работаем на CPU.")

DEVICE = "cuda" if cuda_ok else "cpu"

print("Загрузка модели эмбеддингов...")
embed_model = SentenceTransformer('intfloat/multilingual-e5-large', device=DEVICE)

# ==========================================================
# 2. РЕЖИМ РАБОТЫ (ПЕРЕКЛЮЧАТЕЛЬ)
# ==========================================================
# "local_db"      -> Искать только в постоянной базе (файлы не парсятся)
# "new_docs_only" -> Спарсить папку raw, искать только в ней, не сохранять на диск
SEARCH_MODE = "local_db" 

print("\n" + "=" * 60)
print(f"РЕЖИМ БАЗЫ ДАННЫХ: {SEARCH_MODE}")
print("=" * 60)

if SEARCH_MODE == "local_db":
    # ------------------------------------------------------
    # ВЕТКА 1: ПОСТОЯННАЯ БАЗА ДАННЫХ (НА ДИСКЕ)
    # ------------------------------------------------------
    chroma_db_dir = os.path.join(project_root, "data", "chroma_db")
    os.makedirs(chroma_db_dir, exist_ok=True)
    
    # Подключаемся к базе на жестком диске
    persistent_client = chromadb.PersistentClient(path=chroma_db_dir)
    active_collection = persistent_client.get_or_create_collection("materials_db_notebook")
    
    print(f"Успешно подключена локальная база данных из {chroma_db_dir}")
    print(f"Документов (чанков) в базе: {active_collection.count()}")

elif SEARCH_MODE == "new_docs_only":
    # ------------------------------------------------------
    # ВЕТКА 2: ВРЕМЕННАЯ БАЗА (ТОЛЬКО НОВЫЕ ФАЙЛЫ, ОЗУ)
    # ------------------------------------------------------
    # Создаем базу чисто в ОЗУ. Она исчезнет после закрытия ноутбука.
    temp_client = chromadb.Client() 
    try:
        temp_client.delete_collection("temp_session_db")
    except Exception:
        pass
    active_collection = temp_client.create_collection("temp_session_db")
    
    # 2.1 Настройка Docling с GPU
    from docling.document_converter import DocumentConverter, PdfFormatOption
    from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions, AcceleratorDevice
    from docling.datamodel.base_models import InputFormat
    from docling.chunking import HierarchicalChunker

    accelerator_options = AcceleratorOptions(
        num_threads=6, 
        device=AcceleratorDevice.CUDA if cuda_ok else AcceleratorDevice.CPU,
    )
    pipeline_options = PdfPipelineOptions()
    pipeline_options.accelerator_options = accelerator_options
    pipeline_options.do_ocr = False              
    pipeline_options.do_table_structure = False  

    _converter = DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
    )
    
    reader = GpuDoclingReader()
    reader.converter = _converter
    reader.chunker = HierarchicalChunker()

    # 2.2 Парсинг файлов
    raw_data_dir = os.path.join(project_root, "data", "raw")
    pdf_files = glob.glob(os.path.join(raw_data_dir, "*.pdf"))
    if not pdf_files:
        raise ValueError(f"PDF файлы не найдены в папке {raw_data_dir}!")

    print(f"Начинаем парсинг {len(pdf_files)} документов (без сохранения в основную БД)...")
    documents, metadatas, ids = [], [], []

    for pdf_path in pdf_files:
        fname = os.path.basename(pdf_path)
        t0 = time.time()
        chunks = reader.read(pdf_path)
        dt = time.time() - t0

        speed_flag = "🟢" if dt < 15 else ("🟡" if dt < 60 else "🔴")
        print(f" {speed_flag} {fname}: {dt:.1f} сек, {len(chunks)} чанков")

        for chunk in chunks:
            documents.append(chunk.text)
            ids.append(chunk.chunk_id)
            metadatas.append({
                "source_id": chunk.metadata.source_id,
                "title": chunk.metadata.title,
                "source_type": chunk.metadata.source_type
            })

    # 2.3 Векторизация и запись во временную БД
    print(f"\nИзвлечено {len(documents)} фрагментов. Начинаем векторизацию...")
    t0 = time.time()
    embeddings = embed_model.encode(
        documents,
        batch_size=64 if cuda_ok else 16,
        show_progress_bar=True,
    ).tolist()
    print(f"Векторизация заняла {time.time() - t0:.1f} сек")

    active_collection.add(
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids
    )
    print(" Временная база знаний заполнена и готова к поиску.")

else:
    raise ValueError("Неизвестный SEARCH_MODE! Укажите 'local_db' или 'new_docs_only'.")

ПРОВЕРКА GPU И ЗАГРУЗКА МОДЕЛЕЙ
Устройство: NVIDIA A100 80GB PCIe MIG 2g.20gb
Загрузка модели эмбеддингов...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


РЕЖИМ БАЗЫ ДАННЫХ: local_db
✅ Успешно подключена локальная база данных из /home/jovyan/persistent_volume/FACRORY/hypothesis-factory/data/chroma_db
Документов (чанков) в базе: 2358


In [3]:
# ==========================================================
# УТИЛИТА: ДОБАВЛЕНИЕ НОВЫХ PDF В ПОСТОЯННУЮ БАЗУ ДАННЫХ
# ==========================================================
import os
import sys
import glob
import time
import torch
import chromadb
from sentence_transformers import SentenceTransformer

project_root = os.path.abspath('..')
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

from hypothesis_factory.ingestion_gpu import GpuDoclingReader

# Настройка путей
chroma_db_dir = os.path.join(project_root, "data", "chroma_db")
raw_data_dir = os.path.join(project_root, "data", "raw_to_add") # Отдельная папка для новых файлов!

# Создадим папку для новых файлов, если её нет
os.makedirs(raw_data_dir, exist_ok=True)

# 1. Проверяем наличие файлов для добавления
pdf_files = glob.glob(os.path.join(raw_data_dir, "*.pdf"))
if not pdf_files:
    print(f" Папка '{raw_data_dir}' пуста.")
    print("Положите туда новые PDF файлы и запустите ячейку снова.")
    sys.exit()

print(f"Найдено {len(pdf_files)} новых файлов для добавления в базу.")

# 2. Инициализируем GPU и модели
cuda_ok = torch.cuda.is_available()
DEVICE = "cuda" if cuda_ok else "cpu"
print(f"Загрузка модели эмбеддингов на {DEVICE}...")
embed_model = SentenceTransformer('intfloat/multilingual-e5-large', device=DEVICE)

# 3. Подключаемся к существующей базе данных (или создаем, если первый раз)
print("\nПодключение к ChromaDB...")
persistent_client = chromadb.PersistentClient(path=chroma_db_dir)
collection = persistent_client.get_or_create_collection("materials_db_notebook")
initial_count = collection.count()
print(f"Текущее количество чанков в базе: {initial_count}")

# 4. Настраиваем парсер (Docling)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions, AcceleratorDevice
from docling.datamodel.base_models import InputFormat
from docling.chunking import HierarchicalChunker

accelerator_options = AcceleratorOptions(
    num_threads=6, 
    device=AcceleratorDevice.CUDA if cuda_ok else AcceleratorDevice.CPU,
)
pipeline_options = PdfPipelineOptions()
pipeline_options.accelerator_options = accelerator_options
pipeline_options.do_ocr = False              
pipeline_options.do_table_structure = False  

_converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

reader = GpuDoclingReader()
reader.converter = _converter
reader.chunker = HierarchicalChunker()

# 5. Парсинг и добавление
documents, metadatas, ids = [], [], []

print("\nНачинаем парсинг...")
for pdf_path in pdf_files:
    fname = os.path.basename(pdf_path)
    t0 = time.time()
    
    try:
        chunks = reader.read(pdf_path)
        dt = time.time() - t0
        
        # Генерируем уникальный префикс для ID чанков этого файла, 
        # чтобы избежать коллизий с уже существующими в базе
        file_hash = str(hash(fname))[-6:] 
        
        for chunk in chunks:
            documents.append(chunk.text)
            # Делаем ID уникальными: старый_id + хэш_файла
            ids.append(f"{chunk.chunk_id}_{file_hash}")
            metadatas.append({
                "source_id": fname, # Записываем имя файла для цитирования
                "title": chunk.metadata.title or fname,
                "source_type": chunk.metadata.source_type
            })
            
        print(f"  {fname}: {dt:.1f} сек, извлечено {len(chunks)} чанков")
        
        # Опционально: переместить файл после успешного парсинга, чтобы не парсить дважды
        # os.rename(pdf_path, os.path.join(project_root, "data", "processed", fname))
        
    except Exception as e:
        print(f" Ошибка при парсинге {fname}: {str(e)}")

if not documents:
    print("Не удалось извлечь текст ни из одного файла.")
    sys.exit()

# 6. Векторизация и сохранение
print(f"\nНачинаем векторизацию {len(documents)} новых фрагментов...")
t0 = time.time()
embeddings = embed_model.encode(
    documents,
    batch_size=64 if cuda_ok else 16,
    show_progress_bar=True,
).tolist()
print(f"Векторизация заняла {time.time() - t0:.1f} сек")

print("Запись в базу данных...")
collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

final_count = collection.count()
print("\n" + "=" * 60)
print(" ГОТОВО! БАЗА ДАННЫХ ОБНОВЛЕНА")
print("=" * 60)
print(f"Было чанков:  {initial_count}")
print(f"Добавлено:    {len(documents)}")
print(f"Стало чанков: {final_count}")

Найдено 1 новых файлов для добавления в базу.
Загрузка модели эмбеддингов на cuda...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Подключение к ChromaDB...
Текущее количество чанков в базе: 0

Начинаем парсинг...


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

 ✅ geokniga-metallurgiya-blagorodnyh-metallov_0.pdf: 33.8 сек, извлечено 2358 чанков

Начинаем векторизацию 2358 новых фрагментов...


Batches:   0%|          | 0/37 [00:00<?, ?it/s]

Векторизация заняла 27.5 сек
Запись в базу данных...

 ГОТОВО! БАЗА ДАННЫХ ОБНОВЛЕНА
Было чанков:  0
Добавлено:    2358
Стало чанков: 2358


In [9]:
# Тест оркестра


# ==========================================================
# 1. Импорты и настройка
# ==========================================================
import logging

# Строгие импорты из пакета hypothesis_factory
from hypothesis_factory.base import BusinessRequest, DocumentChunk, DocumentMetadata
from hypothesis_factory.generator import YandexGPTGenerator
from hypothesis_factory.critic import YandexGPTCritic
from hypothesis_factory.pipeline import HypothesisRefinementPipeline

# Настроим логирование, чтобы видеть процесс общения Генератора и Критика прямо в ячейке
logging.getLogger().setLevel(logging.INFO)

# ==========================================================
# 2. Формируем бизнес-запрос
# ==========================================================
print("=== ШАГ 1: Формирование бизнес-запроса ===")
request = BusinessRequest(
    target_kpi="Повысить предел прочности сплава на 15% при сохранении пластичности.",
    constraints=[
        "Запрещено использование токсичных или радиоактивных элементов",
        "Бюджет на экспериментальную партию не более 500 000 рублей",
        "Технология должна быть применима на стандартном прокатном стане"
    ]
)
print(f"Цель: {request.target_kpi}")
print(f"Ограничения: {request.constraints}\n")

# ==========================================================
# 3. Поиск релевантного контекста (Retriever)
# ==========================================================
print("=== ШАГ 2: Векторный поиск по базе знаний (ChromaDB) ===")
# Формируем строку запроса для поиска на основе цели и ограничений
query_text = f"{request.target_kpi} Ограничения: {', '.join(request.constraints)}"
query_embedding = embed_model.encode([query_text]).tolist()

# Ищем топ-7 самых релевантных кусков текста
results = collection.query(
    query_embeddings=query_embedding,
    n_results=7 
)

context_chunks = []
if results['ids'] and results['ids'][0]:
    for i in range(len(results['ids'][0])):
        chunk_id = results['ids'][0][i]
        text = results['documents'][0][i]
        meta = results['metadatas'][0][i]

        # Преобразуем данные из ChromaDB в наши Pydantic контракты
        metadata = DocumentMetadata(
            source_id=meta.get("source_id", f"unknown_doc_{i}"),
            source_type=meta.get("source_type", "article"),
            title=meta.get("title", "Неизвестный источник")
        )
        context_chunks.append(DocumentChunk(chunk_id=chunk_id, text=text, metadata=metadata))

print(f"Найдено релевантных фрагментов: {len(context_chunks)}\n")

# ==========================================================
# 4. Запуск Оркестратора (Self-Correction Loop) с закрытием сессий
# ==========================================================
print("=== ШАГ 3: Запуск LLM-пайплайна (Генератор + Критик) ===")

# Используем контекстные менеджеры, чтобы гарантировать закрытие API-сессий
with YandexGPTGenerator() as generator, YandexGPTCritic() as critic:
    
    # Собираем оркестратор
    pipeline = HypothesisRefinementPipeline(
        generator=generator, 
        critic=critic,
        min_overall_score=7.0, 
        max_iterations=3
    )

    # Запускаем цикл (пока мы внутри with, API-соединения активны)
    final_hypotheses = pipeline.run_loop(request=request, context=context_chunks)

# Как только код выходит из блока with, автоматически вызываются generator.close() и critic.close()
print("API-сессии успешно закрыты.")

# ==========================================================
# 5. Красивый вывод результатов
# ==========================================================
print("\n" + "="*80)
print(f" ИТОГОВЫЙ ОТЧЕТ: СГЕНЕРИРОВАНО ГИПОТЕЗ - {len(final_hypotheses)}")
print("="*80 + "\n")

for i, hyp in enumerate(final_hypotheses, 1):
    status = " ВАЛИДНА" if hyp.overall_score >= pipeline.min_overall_score else "⚠️ НИЖЕ ПОРОГА"
    
    print(f"[{i}] {hyp.title.upper()} ({status})")
    print(f"Рейтинг: {hyp.overall_score}/10 (Новизна: {hyp.novelty_score}, Реализуемость: {hyp.feasibility_score})")
    print(f"Суть: {hyp.text}")
    print(f"Обоснование: {hyp.reasoning}")
    print(f"Механизм: {hyp.mechanism}")
    
    if hyp.technical_risks:
        print("Технические риски:")
        for risk in hyp.technical_risks:
            print(f"  - {risk}")
            
    if hyp.economic_risks:
        print("Экономические риски:")
        for risk in hyp.economic_risks:
            print(f"  - {risk}")
            
    if hyp.source_refs:
        print(f"Источники: {', '.join(hyp.source_refs)}")
    else:
        print("Источники: Базовые знания модели (Вне контекста)")
        
    print("-" * 80 + "\n")

=== ШАГ 1: Формирование бизнес-запроса ===
Цель: Повысить предел прочности сплава на 15% при сохранении пластичности.
Ограничения: ['Запрещено использование токсичных или радиоактивных элементов', 'Бюджет на экспериментальную партию не более 500 000 рублей', 'Технология должна быть применима на стандартном прокатном стане']

=== ШАГ 2: Векторный поиск по базе знаний (ChromaDB) ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

API call failed on attempt 1: Error code: 403 - {'error': {'message': 'rpc error: code = PermissionDenied desc = Permission denied', 'type': 'permission_error'}}
Max retries exceeded. Total attempts: 1, Last error: Error code: 403 - {'error': {'message': 'rpc error: code = PermissionDenied desc = Permission denied', 'type': 'permission_error'}}
Критический сбой при генерации гипотез: Error code: 403 - {'error': {'message': 'rpc error: code = PermissionDenied desc = Permission denied', 'type': 'permission_error'}}
Генератор не смог предложить базовый набор гипотез.


Найдено релевантных фрагментов: 7

=== ШАГ 3: Запуск LLM-пайплайна (Генератор + Критик) ===
API-сессии успешно закрыты.

 ИТОГОВЫЙ ОТЧЕТ: СГЕНЕРИРОВАНО ГИПОТЕЗ - 0

